In [ ]:
import os
import json
import time
import hashlib
import requests
from itertools import cycle
from google.colab import userdata

# Firebase libraries
import firebase_admin
from firebase_admin import credentials, firestore

# ==========================================
# 1. SETUP & CONNECTIONS
# ==========================================

# Hugging Face Token (Colab Secrets se)
HF_API_TOKEN = userdata.get('HF_TOKEN')

# NOTE: old api-inference.huggingface.co subdomain is dead.
# New router auto-selects a provider that actually serves the model.
API_URL = "https://router.huggingface.co/v1/chat/completions"
MODEL_NAME = "moonshotai/Kimi-K2-Instruct-0905"  # confirmed working

headers = {"Authorization": f"Bearer {HF_API_TOKEN}"}

# Firebase Setup
if not firebase_admin._apps:
    cred = credentials.Certificate("firebase_key.json")
    firebase_admin.initialize_app(cred)

db = firestore.client()

# ==========================================
# 2. CATEGORIES SETUP
# ==========================================
categories = [
    "Deep Philosophical and Aesthetic",
    "Sigma Stoic Mindset",
    "Funny Tech and Programming",
    "Gym and Hustle Motivation"
]
category_cycler = cycle(categories)

# ==========================================
# 3. AI GENERATION FUNCTION (UPDATED for chat completions API)
# ==========================================
def generate_quotes(category, num_quotes=3):
    prompt = f"""Act as an expert Instagram creator. Generate {num_quotes} highly engaging, fresh, and viral {category} quotes.
Return ONLY a valid JSON array of objects with "quote" and "author" keys.
Do not add any explanations, intro text, or markdown."""

    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 1000,
        "temperature": 0.8
    }

    try:
        response = requests.post(API_URL, headers=headers, json=payload, timeout=20)

        print(f"Status Code: {response.status_code}")

        if response.status_code != 200:
            print(f"API Error: {response.status_code} - {response.text}")
            return []

        res_json = response.json()
        raw_text = res_json["choices"][0]["message"]["content"].strip()

        # Cleaning logic
        if raw_text.startswith("```json"):
            raw_text = raw_text.replace("```json", "").replace("```", "").strip()
        elif raw_text.startswith("```"):
            raw_text = raw_text.replace("```", "").strip()

        parsed_quotes = json.loads(raw_text)
        return parsed_quotes

    except Exception as e:
        print(f"   ⚠️ Connection/Processing Error: {e}")
        return []

    return []

# ==========================================
# 4. MAIN INFINITE LOOP
# ==========================================
print("\n--- Starting Infinite Quote Generator ---")

try:
    while True:
        current_category = next(category_cycler)
        print(f"➤ Generating for: {current_category}...")

        new_quotes = generate_quotes(current_category)

        if not new_quotes:
            print("   ⚠️ Retrying in 15 seconds...")
            time.sleep(15)
            continue

        saved_count = 0
        for item in new_quotes:
            quote_text = item.get("quote", "").strip()
            author_name = item.get("author", "Unknown").strip()

            if quote_text:
                unique_id = hashlib.md5(quote_text.lower().encode('utf-8')).hexdigest()

                data_to_store = {
                    "quote_id": unique_id,
                    "category": current_category,
                    "quote": quote_text,
                    "author": author_name,
                    "is_posted": False,
                    "timestamp": firestore.SERVER_TIMESTAMP
                }

                db.collection("instagram_quotes").document(unique_id).set(data_to_store, merge=True)
                saved_count += 1

        print(f"   ✓ {saved_count} quotes saved.")

        # API aur Network ko saans lene ke liye delay
        time.sleep(10)

except KeyboardInterrupt:
    print("\n--- Process Stopped ---")

KeyboardInterrupt: 

In [ ]:
!ping -c 3 api-inference.huggingface.co

/bin/bash: line 1: ping: command not found


In [ ]:
import firebase_admin
from firebase_admin import credentials, firestore

# Agar database connection pehle se active nahi hai, toh in 2 lines ko use karein:
# cred = credentials.Certificate("firebase_key.json")
# firebase_admin.initialize_app(cred)

db = firestore.client()

# 'instagram_quotes' collection ka total count fetch karna
collection_ref = db.collection("instagram_quotes")
count_query = collection_ref.count()
results = count_query.get()

# Exact number print karna
total_quotes = results[0][0].value
print(f"🎉 Firebase mein is waqt total {total_quotes} quotes save ho chuke hain!")

🎉 Firebase mein is waqt total 166 quotes save ho chuke hain!
